# Compra de carros: XGBoost e SVM
**Marlon Nascimento · Projeto de Parceria EBAC / Semantix · Módulo 41**

Consolidação dos exercícios M39 e M40. O objetivo é distinguir compradores registrados na base e discutir uma hipótese de priorização comercial. O histórico não prova compra futura ou incremento de vendas. A documentação completa está em `docs/PROJETO.md`.

## Coleta de dados
Fonte: [Cars - Purchase Decision Dataset / Gabriel Santello](https://www.kaggle.com/datasets/gabrielsantello/cars-purchase-decision-dataset), Kaggle. Licença CC0 confirmada nos metadados públicos em 30/09/2026. A cópia baixada coincide com os exemplos e correlações do M40; o CSV original do curso estava ausente. País, moeda, período de observação e independência dos perfis não foram confirmados.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'src' / 'analise.py').exists():
    root = root.parent
sys.path.insert(0, str(root))
from src.analise import load_data, run
from IPython.display import display, Image
import pandas as pd
raw, X, y, sha = load_data()
print('Dimensões:', raw.shape, '| Ausentes:', int(raw.isna().sum().sum()))
print('SHA256:', sha)
display(raw.drop(columns='User ID').describe(include='all').T)
display(y.value_counts().sort_index().to_frame('clientes'))

Dimensões: (1000, 5) | Ausentes: 0
SHA256: 24db0b2bd9b47b15f55fb6ba0a4c59177f2ec9751eda24f3abfbbef4a012c244


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Gender,1000,2,Female,516,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,1000.0,NaN,NaN,NaN,40.106,10.707073,18.0,32.0,40.0,48.0,63.0
AnnualSalary,1000.0,NaN,NaN,NaN,72689.0,34488.341867,15000.0,46375.0,72000.0,90000.0,152500.0
Purchased,1000.0,NaN,NaN,NaN,0.402,0.490547,0.0,0.0,0.0,1.0,1.0


,clientes
Purchased,
0,598
1,402


## Modelagem
Os três modelos do M40 e seus parâmetros são preservados. A codificação de gênero é fixa e o identificador é excluído. SVMs usam `StandardScaler` em `Pipeline`, ajustado somente no treino em cada partição. XGBoost recebe a escala original. O script abaixo exporta também partições e previsões.

A referência majoritária e os cinco folds estratificados no treino complementam a avaliação. O maior F1 médio indica o candidato antes da reavaliação do teste histórico. Não há busca de hiperparâmetros. O teste contém 200 registros e já havia sido usado no exercício original.

In [2]:
resultados, metricas, folds = run()
display(metricas.round(4))
display(pd.read_csv(root / 'reports' / 'validacao_resumo.csv').round(4))
print('Perfis do teste também presentes no treino:', resultados['perfis_X_teste_presentes_treino'])

                Modelo  Acurácia  Precisão  Recall     F1  ROC_AUC  Balanced_Accuracy
Referência majoritária     0.600    0.0000  0.0000 0.0000   0.5000             0.5000
            SVM Linear     0.855    0.8592  0.7625 0.8079   0.9239             0.8396
        SVM Polinomial     0.875    0.9104  0.7625 0.8299   0.9310             0.8562
               XGBoost     0.920    0.9000  0.9000 0.9000   0.9662             0.9167

CV F1:
                Modelo   mean    std
Referência majoritária 0.0000 0.0000
            SVM Linear 0.7440 0.0482
        SVM Polinomial 0.7757 0.0613
               XGBoost 0.8864 0.0403

Modelo indicado: XGBoost | SHA256: 24db0b2bd9b47b15f55fb6ba0a4c59177f2ec9751eda24f3abfbbef4a012c244


,Modelo,Acurácia,Precisão,Recall,F1,ROC_AUC,Balanced_Accuracy
0,Referência majoritária,0.600,0.0000,0.0000,0.0000,0.5000,0.5000
1,SVM Linear,0.855,0.8592,0.7625,0.8079,0.9239,0.8396
2,SVM Polinomial,0.875,0.9104,0.7625,0.8299,0.9310,0.8562
3,XGBoost,0.920,0.9000,0.9000,0.9000,0.9662,0.9167


,Modelo,mean,std
0,Referência majoritária,0.0000,0.0000
1,SVM Linear,0.7440,0.0482
2,SVM Polinomial,0.7757,0.0613
3,XGBoost,0.8864,0.0403


Perfis do teste também presentes no treino: 21


### Análise exploratória
Idade e renda mostram associação com a compra. A distribuição das classes explica por que a acurácia trivial de 60% é insuficiente para identificar compradores. A moeda da renda não foi confirmada; as correlações não demonstram causalidade.

In [3]:
display(Image(filename=str(root / 'reports/figures/01_eda.png')))

![Visualização agregada](../reports/figures/01_eda.png)

### Comparação e avaliação
Precisão, recall e F1 referem-se à classe `Purchased=1`. ROC-AUC avalia a ordenação dos escores. O gráfico usa o mesmo teste para os modelos e registra a nova execução.

In [4]:
display(Image(filename=str(root / 'reports/figures/02_comparacao.png')))
display(Image(filename=str(root / 'reports/figures/03_erros.png')))
display(Image(filename=str(root / 'reports/figures/04_validacao.png')))

![Visualização agregada](../reports/figures/02_comparacao.png)

![Visualização agregada](../reports/figures/03_erros.png)

![Visualização agregada](../reports/figures/04_validacao.png)

### Auditoria histórica
O M40 registrava 91,5% de acurácia, F1 0,8944 e ROC-AUC 0,9666 para XGBoost. A nova execução registra 92,0%, F1 0,9000 e ROC-AUC 0,9662. Os SVMs coincidem a quatro casas decimais. O ambiente antigo não foi registrado e o CSV do curso não estava disponível; não afirmamos uma causa comprovada para a diferença. Os resultados originais permanecem nos notebooks preservados.

O M39 configurava 100 árvores e learning rate 0,10; o M40 passou a 150 e 0,05. Reproduzimos o M40 e corrigimos a afirmação antiga de que as duas configurações seriam idênticas.

## Conclusões
O XGBoost obtém o maior F1 médio da validação (0,8864) e F1 de teste 0,9000. Identifica 72 dos 80 compradores, contra 61 dos SVMs; deixa oito compradores sem identificação e seleciona oito não compradores. A maior precisão do SVM polinomial (0,9104) envolve menor recall (0,7625).

A recomendação é validar um piloto com dados atuais, janela futura e grupo de controle. Onze compradores adicionais identificados não significam onze vendas adicionais. Não foram medidos custos, margem, calibração externa ou desempenho em produção. Perfis repetidos, população desconhecida, poucos atributos e teste histórico limitam a conclusão. Validação agrupada, temporal e comparação sem gênero são próximos passos, não análises já executadas.

In [5]:
display(Image(filename=str(root / 'reports/figures/05_resumo.png')))
print('Versões usadas:', resultados['versoes'])

Versões usadas: {'python': '3.12.14', 'numpy': '2.2.6', 'pandas': '2.2.3', 'sklearn': '1.7.2', 'xgboost': '3.0.5', 'matplotlib': '3.10.7'}


![Visualização agregada](../reports/figures/05_resumo.png)

## Reprodução e entrega
Execute `python src/analise.py` na raiz do repositório após instalar as versões em `requirements-lock.txt`. Resultados, previsões, partições e figuras são salvos em `reports/`. A fonte está em `data/README.md`, a documentação obrigatória em `docs/PROJETO.md` e o roteiro de publicação e submissão em `docs/ENTREGA.md`.

Referências técnicas: [scikit-learn: prevenção de vazamento](https://scikit-learn.org/stable/common_pitfalls.html), [validação cruzada](https://scikit-learn.org/stable/modules/cross_validation.html) e [XGBoost: parâmetros](https://xgboost.readthedocs.io/en/stable/parameter.html).